# Data Visualization — Quiz (Lectures 1–7)

**Time:** 90 minutes
**Tools:** Plotly only (`plotly.express` first, then `update_traces` / `update_layout` / `update_xaxes` / `update_yaxes`)

**Your task:** answer each analytical question with **one polished, explanatory visualization**. For every question **you decide which chart type fits best** — and justify the choice in one or two sentences.

**Every figure must have:**
1. An **insight title** that states the finding, not just the variables
2. `template='plotly_white'` and a decluttered layout (no unnecessary gridlines, legends or axis titles)
3. A **CVD-safe palette** — no red/green; grey for context + one accent colour where it helps
4. Colour that **encodes meaning**, not decoration
5. Direct labels or annotations where they help the reader

**Chart types covered in the course** — choose from these:
line chart · sorted bar chart · stacked / 100% stacked bar chart · scatter plot · bubble chart · histogram · box plot · violin plot · choropleth map · treemap

> 💡 Each question has one chart type that fits it best, and the questions together cover most of the list above. If two of your answers use the same chart type, check the question again.


**Submit:** this notebook pushed to your personal repo as `quiz/dataviz_quiz.ipynb`

## The dataset — World Population 1970–2022

**Source:** [World Population Dataset](https://www.kaggle.com/datasets/iamsouravbanerjee/world-population-dataset) on Kaggle, compiled from Worldometer (UN World Population Prospects estimates). **234 rows × 16 columns — one row per country or territory, no missing values.**

| Column | Type | Description |
|---|---|---|
| `Country`, `Capital` | categorical | Country / territory and its capital |
| `Continent` | categorical | Africa, Asia, Europe, North America, Oceania, South America |
| `CCA3` | spatial | ISO-3 country code — use with `locationmode='ISO-3'` |
| `1970 Population` … `2022 Population` | temporal | Population in 1970, 1980, 1990, 2000, 2010, 2015, 2020, 2022 |
| `Area` | numerical | Land area in km² |
| `Density` | numerical | People per km² (2022) |
| `World share (%)` | numerical | Share of world population in 2022 |
| `Rank` | numerical | Rank by 2022 population |

**The next setup cell also gives you `df_long` — the same data in **tidy (long) format** with one row per country per year. Use `df` for one-year questions and `df_long` for questions about change over time**

In [1]:
import pandas as pd
import plotly.express as px
import numpy as np

df = pd.read_csv('world_population.csv')

# Shorter column names
df = df.rename(columns={'Country/Territory': 'Country', 'Area (km²)': 'Area',
                        'Density (per km²)': 'Density', 'World Population Percentage': 'World share (%)'})

# Tidy (long) version: one row per country per year
year_cols = [c for c in df.columns if c.endswith('Population')]
df_long = df.melt(id_vars=['CCA3', 'Country', 'Continent'], value_vars=year_cols,
                  var_name='Year', value_name='Population')

df_long['Year'] = df_long['Year'].str[:4].astype(int)     # '2022 Population' → 2022

print(df.shape, df_long.shape)


(234, 16) (1872, 5)


In [2]:
# Shared CVD-safe palette (Okabe–Ito) used across all figures
GREY       = '#BBBBBB'   # context
DARK_GREY  = '#555555'   # text / annotations
ACCENT     = '#E66100'   # orange accent (the finding)
BLUE       = '#0072B2'   # second accent (contrast)

# One consistent colour per continent (only used where continent itself is the encoding)
CONT_COLORS = {'Asia': '#0072B2', 'Africa': '#E69F00', 'Europe': '#56B4E9',
               'North America': '#009E73', 'South America': '#CC79A7', 'Oceania': '#999999'}

BASE_LAYOUT = dict(template='plotly_white', font=dict(family='Arial', size=13, color=DARK_GREY),
                   title_font=dict(size=17, color='#222222'), margin=dict(l=60, r=40, t=90, b=50))


---
## Q1 — Which continent is growing fastest?

How has the population of each continent **grown since 1970, relative to its 1970 size** — and which continent is growing fastest?


**Chart choice & justification:** **Line chart of a growth index (1970 = 100).**

The question is about *change over time* and *relative* growth, so a line chart with a numeric year axis (which keeps the uneven 1970–2022 spacing honest) is the natural choice. Indexing every continent to 1970 = 100 puts them on a common scale — raw totals would make Asia dominate and hide Oceania's or Africa's pace.

*Design choices:* Africa is the only coloured line (orange accent); the other five continents are grey context. Lines are labelled directly at their right end (no legend), y-gridlines are kept light for reading values, x-gridlines are removed, and a dotted reference line marks "no growth" (100). The title states the answer.

**Answer:** **Africa** is growing fastest — its population is **3.9×** its 1970 size (index ≈ 390). Oceania (2.3×), South America (2.3×) and Asia (2.2×) roughly doubled, North America grew 1.9×, while Europe grew only 13%.

In [3]:
# Q1 — continent growth relative to 1970
cont = df_long.groupby(['Continent', 'Year'], as_index=False)['Population'].sum()
base = cont[cont['Year'] == 1970].set_index('Continent')['Population']
cont['Index'] = cont['Population'] / cont['Continent'].map(base) * 100

color_map = {c: (ACCENT if c == 'Africa' else GREY) for c in cont['Continent'].unique()}

fig = px.line(cont, x='Year', y='Index', color='Continent', color_discrete_map=color_map,
              title="<b>Africa's population has nearly quadrupled since 1970 — the fastest of any continent</b>"
                    "<br><sup>Population index, 1970 = 100</sup>",
              hover_data={'Population': ':,.0f', 'Index': ':.0f'})

fig.update_traces(line=dict(width=2))
fig.update_traces(line=dict(width=4), selector=dict(name='Africa'))

# Direct labels at line ends instead of a legend
last = cont[cont['Year'] == 2022].sort_values('Index')
y_pos = {'Oceania': 245, 'South America': 228, 'Asia': 211, 'North America': 190, 'Europe': 113, 'Africa': 390}
for _, r in last.iterrows():
    is_af = r['Continent'] == 'Africa'
    fig.add_annotation(x=2022, y=y_pos[r['Continent']], text=f"<b>{r['Continent']}</b> {r['Index']/100:.1f}×" if is_af
                       else f"{r['Continent']} {r['Index']/100:.1f}×",
                       xanchor='left', xshift=6, showarrow=False,
                       font=dict(color=ACCENT if is_af else DARK_GREY, size=13 if is_af else 12))

fig.add_hline(y=100, line_dash='dot', line_color=GREY)
fig.update_layout(**BASE_LAYOUT, showlegend=False, height=520)
fig.update_layout(margin=dict(r=160))
fig.update_xaxes(title=None, showgrid=False, tickvals=[1970, 1980, 1990, 2000, 2010, 2020])
fig.update_yaxes(title=None, gridcolor='#EEEEEE', ticksuffix='', range=[80, 420])
fig.show()

---
## Q2 — Which countries added the most people?

Which **15 countries added the most people** between 1970 and 2022 — and how many of them are in **Africa**?

**Chart choice & justification:** **Sorted horizontal bar chart.**

This is a ranking of one quantity (people added) across 15 named categories, so a sorted bar chart is the most accurate choice: bar length is the easiest visual encoding to compare, sorting makes the rank obvious, and horizontal bars give room for country names without rotating text.

*Design choices:* colour encodes the second part of the question — **African countries are orange, all others grey** — so the reader can count them at a glance. Each bar has a direct value label (millions), so the x-axis, its gridlines and the legend are removed; a small colour key sits in the subtitle.

**Answer:** India (+860 M) and China (+603 M) added by far the most people, followed by Pakistan, Nigeria and Indonesia. **4 of the top 15 are in Africa:** Nigeria (4th overall, +163 M), Ethiopia (+95 M), DR Congo (+79 M) and Egypt (+76 M). The other 11 are in Asia (8) and the Americas (3).

In [4]:
# Q2 — top 15 countries by absolute population added 1970 → 2022
q2 = df[['Country', 'Continent', '1970 Population', '2022 Population']].copy()
q2['Added'] = q2['2022 Population'] - q2['1970 Population']
top15 = q2.nlargest(15, 'Added').sort_values('Added')
top15['Group'] = np.where(top15['Continent'] == 'Africa', 'Africa', 'Other continents')
top15['Label'] = (top15['Added'] / 1e6).round(0).astype(int).map('+{:,} M'.format)
n_africa = (top15['Group'] == 'Africa').sum()

fig = px.bar(top15, x='Added', y='Country', orientation='h', color='Group', text='Label',
             color_discrete_map={'Africa': ACCENT, 'Other continents': GREY},
             title=f"<b>{n_africa} of the 15 countries that added the most people since 1970 are in Africa</b>"
                   f"<br><sup>People added 1970–2022 · <span style='color:{ACCENT}'><b>Africa</b></span> vs "
                   f"<span style='color:#999999'><b>other continents</b></span></sup>",
             hover_data={'Continent': True, 'Added': ':,.0f', 'Group': False, 'Label': False})

fig.update_traces(textposition='outside', cliponaxis=False, textfont=dict(color=DARK_GREY))
fig.update_layout(**BASE_LAYOUT, showlegend=False, height=560, bargap=0.25)
fig.update_xaxes(visible=False, showgrid=False)
fig.update_yaxes(title=None, showgrid=False, categoryorder='total ascending')
fig.show()

---
## Q3 — Do bigger countries have more people?

Is there a relationship between a country's **land area** and its **2022 population** — does it differ by **continent**, and which countries clearly break the pattern?

**Chart choice & justification:** **Scatter plot on log–log axes.**

The question asks about the *relationship between two numerical variables* (area and population), which is exactly what a scatter plot shows. Both variables span 6–9 orders of magnitude (Vatican 1 km² → Russia 17 M km²), so log scales are essential — on linear axes almost all 234 points would collapse into one corner.

*Design choices:* colour encodes **continent** (CVD-safe Okabe–Ito palette, same colours as the treemap in Q6) so continental differences are visible; a dashed grey trend line (log–log least-squares fit) shows the overall pattern; the countries that break the pattern most are labelled directly. Axis tick labels use readable units (k, M).

**Answer:** Yes — there is a **strong positive relationship** (correlation of the logs ≈ **0.87**): bigger countries generally have more people. Asian countries sit mostly **above** the line (more people than their size predicts), while the Americas and Oceania sit mostly **below** it. Clear outliers: **Bangladesh, India, and the city-states Macau, Singapore, Hong Kong and Monaco** have far more people than their area predicts; **Greenland, the Falkland Islands, Iceland, Western Sahara and French Guiana** have far fewer.

In [5]:
# Q3 — area vs 2022 population
q3 = df[['Country', 'Continent', 'Area', '2022 Population']].copy()
lx, ly = np.log10(q3['Area']), np.log10(q3['2022 Population'])
slope, intercept = np.polyfit(lx, ly, 1)
r = np.corrcoef(lx, ly)[0, 1]
q3['Residual'] = ly - (slope * lx + intercept)

fig = px.scatter(q3, x='Area', y='2022 Population', color='Continent', log_x=True, log_y=True,
                 color_discrete_map=CONT_COLORS, hover_name='Country',
                 hover_data={'Area': ':,.0f', '2022 Population': ':,.0f', 'Continent': False},
                 title=f"<b>Bigger countries generally have more people (r = {r:.2f} on log scales) — "
                       f"<br>but Bangladesh and the Asian city-states pack in far more, Greenland far fewer</b>"
                       f"<br><sup>Land area vs 2022 population, both on log scales · dashed line = overall trend</sup>")
fig.update_traces(marker=dict(size=8, opacity=0.8, line=dict(width=0.5, color='white')))

# Trend line
xs = np.logspace(np.log10(q3['Area'].min()), np.log10(q3['Area'].max()), 50)
fig.add_scatter(x=xs, y=10 ** (slope * np.log10(xs) + intercept), mode='lines',
                line=dict(color=DARK_GREY, dash='dash', width=1.5), hoverinfo='skip', showlegend=False)

# Label the countries that break the pattern the most (largest residuals in both directions)
outliers = pd.concat([q3.nlargest(6, 'Residual'), q3.nsmallest(4, 'Residual')])
offsets = {'Macau': (30, -10), 'Singapore': (-70, -25), 'Hong Kong': (60, 25), 'Bangladesh': (-60, -25),
           'India': (-40, -20), 'Monaco': (35, -15), 'Greenland': (-20, 30), 'Falkland Islands': (40, 25),
           'Niue': (25, 25), 'Western Sahara': (-45, 30)}
for _, row in outliers.iterrows():
    ax, ay = offsets.get(row['Country'], (30, -20))
    fig.add_annotation(x=np.log10(row['Area']), y=np.log10(row['2022 Population']), text=row['Country'],
                       ax=ax, ay=ay, arrowhead=0, arrowwidth=1, arrowcolor=DARK_GREY,
                       font=dict(size=11, color='#222222'))

fig.update_layout(**BASE_LAYOUT, height=620,
                  legend=dict(title=None, orientation='h', y=1.0, yanchor='bottom', x=0, font=dict(size=12)))
fig.update_layout(margin=dict(t=140))
fig.update_xaxes(title='Land area (km², log scale)', showgrid=False)
fig.update_yaxes(title='Population 2022 (log scale)', gridcolor='#EEEEEE')
fig.show()

---
## Q4 — How crowded are the continents?

How does **population density vary within and across continents**? Show the spread, not just the average.

**Chart choice & justification:** **Box plot (log scale), one box per continent, sorted by median.**

The question explicitly asks for the **spread within** each continent and the comparison **across** continents. A box plot shows median, interquartile range, whiskers and individual outliers for each group side by side, which is exactly that. Density is extremely skewed (0.03 to 23 000 people/km²), so a log y-axis is needed to see the boxes. With only 6–58 countries per continent, a box plot is more honest than a violin plot (the smoothed density shape would be guessed from few points).

*Design choices:* all boxes share one neutral blue — colour carries no extra meaning here, the categories are already on the axis. Continents are sorted by median density, the most extreme outliers (city-states) are annotated, every country is visible on hover, and the legend is removed.

**Answer:** Density varies **far more within continents than between them**. Asia and Europe each span from ~2–4 people/km² (Mongolia, Iceland) up to >18 000 (Macau, Monaco). Surprisingly **North America has the highest median (~221/km²)** because of its many small, dense Caribbean islands, while **South America is the least crowded and most uniform** continent (median ~18/km², max only 65).

In [6]:
# Q4 — population density by continent
order = df.groupby('Continent')['Density'].median().sort_values(ascending=False).index.tolist()

fig = px.box(df, x='Continent', y='Density', points='all', log_y=True, hover_name='Country',
             category_orders={'Continent': order}, color_discrete_sequence=[BLUE],
             title="<b>Density varies far more within continents than between them —"
                   "<br>Asia and Europe range from under 5 to over 18,000 people per km²</b>"
                   "<br><sup>Population density 2022 (people per km², log scale) · each dot is a country · sorted by median</sup>")
fig.update_traces(marker=dict(size=4, opacity=0.45), jitter=0.35, pointpos=0,
                  line=dict(width=1.5), fillcolor='rgba(0,114,178,0.15)')

for name in ['Macau', 'Monaco', 'Singapore', 'Greenland']:
    row = df[df['Country'] == name].iloc[0]
    fig.add_annotation(x=row['Continent'], y=np.log10(row['Density']), text=name, ax=55, ay=0,
                       arrowhead=0, arrowwidth=1, arrowcolor=DARK_GREY, font=dict(size=11))

# Medians written into the category labels (direct labelling, no overlap with the points)
med = df.groupby('Continent')['Density'].median()
fig.update_xaxes(tickvals=order, ticktext=[f"<b>{c}</b><br>median {med[c]:.0f}" for c in order])

fig.update_layout(**BASE_LAYOUT, showlegend=False, height=580)
fig.update_layout(margin=dict(t=130))
fig.update_xaxes(title=None, showgrid=False)
fig.update_yaxes(title='People per km² (log)', gridcolor='#EEEEEE')
fig.show()

---
## Q5 — Where did populations grow — and where did they shrink?

Which countries' populations **grew and which shrank between 2000 and 2022** (percentage change), and where in the world are they?

**Chart choice & justification:** **Choropleth map with a diverging colour scale centred on 0 %.**

The question asks *where in the world* growth and decline happened, so the geography itself is part of the answer — a choropleth map (using the `CCA3` ISO-3 codes) shows the regional clusters that a bar chart of 234 countries would hide.

*Design choices:* percentage change has a meaningful midpoint (0 = no change), so a **diverging purple–orange scale** (`PuOr`, CVD-safe — no red/green) is used: **orange = shrank, purple = grew**, white ≈ no change. The scale is capped at ±60 % so that a few extreme cases (Qatar +317 %, UAE +188 %) don't wash out every other country; the true value is in the hover. Country borders are thin and light, the frame and coastline clutter are removed.

**Answer:** Between 2000 and 2022, **34 countries/territories shrank**, and **18 of them are in Europe** — a clear cluster across **Eastern Europe and the Balkans** (Lithuania −24 %, Moldova −23 %, Latvia, Ukraine, Bulgaria, Bosnia), plus Japan, Georgia, Cuba/Caribbean islands and several Pacific islands. Growth was strongest across **sub-Saharan Africa** (Niger +125 %) and the **Gulf states** (Qatar +317 %, UAE +188 %).

In [7]:
# Q5 — % population change 2000 → 2022
q5 = df[['CCA3', 'Country', 'Continent', '2000 Population', '2022 Population']].copy()
q5['Change (%)'] = (q5['2022 Population'] / q5['2000 Population'] - 1) * 100
n_shrank = (q5['Change (%)'] < 0).sum()
n_shrank_eu = ((q5['Change (%)'] < 0) & (q5['Continent'] == 'Europe')).sum()

fig = px.choropleth(q5, locations='CCA3', locationmode='ISO-3', color='Change (%)',
                    color_continuous_scale='PuOr', range_color=[-60, 60], hover_name='Country',
                    hover_data={'CCA3': False, 'Change (%)': ':+.1f', '2000 Population': ':,.0f',
                                '2022 Population': ':,.0f'},
                    title=f"<b>{n_shrank} countries shrank between 2000 and 2022 — {n_shrank_eu} of them in Europe, mostly in the East —"
                          f"<br>while Africa and the Gulf grew fastest</b>"
                          f"<br><sup>Population change 2000–2022 (%) · <span style='color:#B35806'><b>orange = shrank</b></span>, "
                          f"<span style='color:#542788'><b>purple = grew</b></span> · scale capped at ±60%</sup>")

fig.update_geos(showframe=False, showcoastlines=False, projection_type='natural earth',
                showcountries=True, countrycolor='#DDDDDD', landcolor='#F2F2F2', lataxis_range=[-58, 85])
fig.update_traces(marker_line_color='white', marker_line_width=0.4)
fig.update_layout(**BASE_LAYOUT, height=560,
                  coloraxis_colorbar=dict(title=None, ticksuffix='%', tickvals=[-60, -30, 0, 30, 60],
                                          ticktext=['≤ −60%', '−30%', '0%', '+30%', '≥ +60%'],
                                          thickness=12, len=0.6))
fig.update_layout(margin=dict(l=10, r=10, t=120, b=10))
fig.show()

---
## Q6 — Where do the world's 8 billion people live?

How is the **2022 world population** distributed across **continents and the countries within them**? Show both levels in one figure.

**Chart choice & justification:** **Treemap (World → Continent → Country).**

The question is about a **part-to-whole** breakdown with **two hierarchical levels** in one figure. A treemap shows both at once: rectangle area is proportional to population, continents are the large blocks and countries are nested inside them. It also handles 234 countries gracefully — tiny territories simply become slivers instead of cluttering a legend (a pie or stacked bar could not do this).

*Design choices:* colour encodes only the **continent** (CVD-safe Okabe–Ito, consistent with Q3), so every country inherits its continent's colour and the hierarchy reads instantly. Labels show the name and the share of world population directly inside each tile; the hover gives exact numbers. No legend is needed because each continent block is labelled.

**Answer:** Of the ~**7.97 billion** people in 2022, **Asia holds 59 %** — and **China and India alone account for ~36 %** of the world. Africa is the second-largest block (~18 %), followed by Europe (~9 %), North America (~7.5 %), South America (~5.5 %) and Oceania (<1 %).

In [8]:
# Q6 — treemap of world population 2022
total = df['2022 Population'].sum()
share_asia = df.loc[df['Continent'] == 'Asia', '2022 Population'].sum() / total * 100
share_ci = df.loc[df['Country'].isin(['China', 'India']), '2022 Population'].sum() / total * 100

fig = px.treemap(df, path=[px.Constant('World'), 'Continent', 'Country'], values='2022 Population',
                 color='Continent', color_discrete_map={**CONT_COLORS, '(?)': '#FFFFFF'},
                 title=f"<b>Asia holds {share_asia:.0f}% of the world's {total/1e9:.1f} billion people — "
                       f"China and India alone account for {share_ci:.0f}%</b>"
                       f"<br><sup>2022 population by continent and country · tile area = population</sup>")

fig.update_traces(texttemplate='<b>%{label}</b><br>%{percentRoot:.1%}',
                  hovertemplate='<b>%{label}</b><br>Population: %{value:,.0f}<br>Share of world: %{percentRoot:.1%}<extra></extra>',
                  marker=dict(line=dict(color='white', width=1)), root_color='#F7F7F7',
                  textfont=dict(color='white'), tiling=dict(pad=2))
fig.update_layout(**BASE_LAYOUT, height=650)
fig.update_layout(margin=dict(l=10, r=10, t=90, b=10))
fig.show()

---
## Q7 — Whose share of the world is rising?

How have the shares of world population held by **Africa, Asia and Europe** changed from 1970 to 2022? Group the other three continents as `'Rest of the world'`.

**Chart choice & justification:** **100 % stacked bar chart (one bar per year).**

The question is about **shares** of a whole changing over time. A 100 % stacked bar fixes every bar at the same total, so the reader compares each group's *proportion* rather than absolute size (which would be dominated by overall world growth). With only four groups and eight time points it stays easy to read.

*Design choices:* colour encodes the story: **Africa (orange accent)** — the rising share — and **Europe (blue)** — the falling share — are the two highlighted groups; Asia is mid-grey and *Rest of the world* is light grey, because their shares barely change. Africa is placed at the bottom (stable baseline) and Europe directly above it so their changing heights are easy to compare. Percentages are printed inside the segments, groups are labelled directly next to the last bar (no legend), and the y-axis is removed.

**Answer:** **Africa's share nearly doubled, from 9.9 % (1970) to 17.9 % (2022)**, while **Europe's share almost halved, from 17.8 % to 9.3 %** — Africa overtook Europe during the 1990s. Asia stayed roughly constant at ~58–60 %, and the rest of the world fell slightly (14.3 % → 13.6 %).

In [9]:
# Q7 — share of world population: Africa, Asia, Europe, Rest of the world
q7 = df_long.copy()
q7['Group'] = q7['Continent'].where(q7['Continent'].isin(['Africa', 'Asia', 'Europe']), 'Rest of the world')
q7 = q7.groupby(['Year', 'Group'], as_index=False)['Population'].sum()
q7['Share'] = q7['Population'] / q7.groupby('Year')['Population'].transform('sum') * 100

af = q7[q7['Group'] == 'Africa'].set_index('Year')['Share']
eu = q7[q7['Group'] == 'Europe'].set_index('Year')['Share']

group_order = ['Africa', 'Europe', 'Asia', 'Rest of the world']
group_colors = {'Africa': ACCENT, 'Europe': BLUE, 'Asia': '#8C8C8C', 'Rest of the world': '#D0D0D0'}

fig = px.bar(q7, x='Year', y='Share', color='Group', text='Share',
             category_orders={'Group': group_order}, color_discrete_map=group_colors,
             hover_data={'Population': ':,.0f', 'Share': ':.1f'},
             title=f"<b>Africa's share of the world nearly doubled ({af[1970]:.1f}% → {af[2022]:.1f}%)"
                   f"<br>while Europe's almost halved ({eu[1970]:.1f}% → {eu[2022]:.1f}%)</b>"
                   f"<br><sup>Share of world population by region, 1970–2022</sup>")

fig.update_traces(texttemplate='%{text:.1f}%', textposition='inside', insidetextanchor='middle',
                  textfont=dict(size=11))
fig.update_traces(textfont_color='#333333', selector=dict(name='Rest of the world'))

# Direct labels next to the last bar
cum = 0
for g in group_order:
    s = q7[(q7['Year'] == 2022) & (q7['Group'] == g)]['Share'].iloc[0]
    # on a category axis a numeric x is the category *position*, so use the index of the last bar
    fig.add_annotation(x=q7['Year'].nunique() - 1, y=cum + s / 2, text=f"<b>{g}</b>", xanchor='left', xshift=32, showarrow=False,
                       font=dict(color=group_colors[g] if g != 'Rest of the world' else '#888888', size=13))
    cum += s

fig.update_layout(**BASE_LAYOUT, barmode='stack', showlegend=False, height=540, bargap=0.25)
fig.update_layout(margin=dict(r=150, t=120))
fig.update_xaxes(title=None, showgrid=False, type='category', categoryorder='array',
                 categoryarray=sorted(q7['Year'].unique()))
fig.update_yaxes(visible=False, range=[0, 100])
fig.show()